In [1]:
"""Locate videos by ID/path fragment and export 64 uniformly sampled frames."""

from pathlib import Path

# Add one or more root directories that contain your videos.
VIDEO_ROOTS = [
    # Path("/path/to/videos"),
    Path("/nas/mars/dataset/MLVU/MLVU/video/"),
    Path("/nas/mars/dataset/Video-MME/burn-subtitles/"),
    Path("/nas/mars/dataset/longvideobench/burn-subtitles/"),
]

# Output directory where sampled frames are written.
OUTPUT_DIR = Path("./video_clip_frames")

# Number of frames to save per video.
NUM_FRAMES = 64

VIDEO_IDS = [
    "@placesunleashed-7321079612488862981",
    "H_b5d-rLXJU",
    "@lisolna-7282789187676294432",
    "5_order/order_165",
    "5_order/order_231",
    "GRPLynULvJY",
    "7_topic_reasoning/movie101_16",
    "6Z7AAcD8rbo",
    "movie.explained6-7269746510462536962",
]

# Optional: add more extensions if your data uses others.
VIDEO_EXTS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v"}


In [2]:
import re
import cv2
import numpy as np


def normalize_fragment(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", value.lower())


def candidate_keys(video_path: Path) -> set[str]:
    rel = video_path.with_suffix("")
    values = {
        str(rel).replace("\\", "/"),
        rel.name,
        video_path.stem,
        video_path.name,
    }

    if len(rel.parts) > 1:
        values.add(str(Path(*rel.parts[-2:])).replace("\\", "/"))

    keys = set(values)
    keys.update(normalize_fragment(v) for v in values)
    return {k for k in keys if k}


def discover_videos(roots: list[Path], exts: set[str]) -> list[Path]:
    videos: list[Path] = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            print(f"[warn] missing root: {root}")
            continue

        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in exts:
                videos.append(path)

    return videos


def find_video_for_id(video_id: str, videos: list[Path]) -> Path | None:
    target = video_id.replace("\\", "/")
    norm_target = normalize_fragment(target)

    for path in videos:
        keys = candidate_keys(path)
        if target in keys or norm_target in keys:
            return path

    for path in videos:
        path_text = str(path.with_suffix("")).replace("\\", "/")
        norm_path_text = normalize_fragment(path_text)
        if target in path_text or norm_target in norm_path_text:
            return path

    return None


def sample_indices(frame_count: int, num_frames: int) -> np.ndarray:
    if frame_count <= 0:
        return np.array([], dtype=np.int32)
    return np.linspace(0, frame_count - 1, num=min(num_frames, frame_count), dtype=np.int32)


def save_uniform_frames(video_path: Path, out_dir: Path, num_frames: int = 64) -> int:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_indices(frame_count, num_frames)

    video_out = out_dir / video_path.stem
    video_out.mkdir(parents=True, exist_ok=True)

    saved = 0
    for i, frame_idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            continue

        cv2.imwrite(str(video_out / f"frame_{i:03d}.jpg"), frame)
        saved += 1

    cap.release()
    return saved


# ---- Run extraction ----
roots = [Path(p) for p in VIDEO_ROOTS]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_videos = discover_videos(roots, VIDEO_EXTS)
print(f"Discovered {len(all_videos)} video files.")

for video_id in VIDEO_IDS:
    match = find_video_for_id(video_id, all_videos)
    if match is None:
        print(f"[missing] {video_id}")
        continue

    frames_saved = save_uniform_frames(match, OUTPUT_DIR, NUM_FRAMES)
    print(f"[ok] {video_id} -> {match} (saved {frames_saved} frames)")

Discovered 3759 video files.
[ok] @placesunleashed-7321079612488862981 -> /nas/mars/dataset/longvideobench/burn-subtitles/@placesunleashed-7321079612488862981.mp4 (saved 64 frames)
[ok] H_b5d-rLXJU -> /nas/mars/dataset/longvideobench/burn-subtitles/H_b5d-rLXJU.mp4 (saved 64 frames)
[ok] @lisolna-7282789187676294432 -> /nas/mars/dataset/longvideobench/burn-subtitles/@lisolna-7282789187676294432.mp4 (saved 64 frames)
[ok] 5_order/order_165 -> /nas/mars/dataset/MLVU/MLVU/video/5_order/order_165.mp4 (saved 64 frames)
[ok] 5_order/order_231 -> /nas/mars/dataset/MLVU/MLVU/video/5_order/order_231.mp4 (saved 64 frames)
[ok] GRPLynULvJY -> /nas/mars/dataset/longvideobench/burn-subtitles/GRPLynULvJY.mp4 (saved 64 frames)
[ok] 7_topic_reasoning/movie101_16 -> /nas/mars/dataset/MLVU/MLVU/video/7_topic_reasoning/movie101_16.mp4 (saved 64 frames)
[ok] 6Z7AAcD8rbo -> /nas/mars/dataset/longvideobench/burn-subtitles/6Z7AAcD8rbo.mp4 (saved 64 frames)
[ok] movie.explained6-7269746510462536962 -> /nas/mar